# Day 5.8 — Pivotal Exercise: Build a Capability-Aware Tool Registry

This is the day's one hands-on implementation lab. It uses no API key. Try the starter cell first; a fully commented reference solution follows the check so you can compare or catch up.

## Why this mechanism matters

A harness needs one controlled place for tool discovery and dispatch. The registry connects model-visible schemas to host-owned handlers while policy limits which capabilities a configuration receives.

## Contract

Reject duplicate registrations with `ValueError`. Show schemas only for granted capabilities. Raise `KeyError` for an unknown tool and `PermissionError` for an ungranted one before the handler is called.

Before coding, write one sentence predicting the easiest mistake to make.

In [ ]:
class ToolRegistry:
    def __init__(self):
        self._tools = {}   # name -> {"schema": ..., "handler": ..., "capability": ...}

    def register(self, name, schema, handler, capability):
        """Register once; a second registration of the same name raises ValueError."""
        raise NotImplementedError("Complete registration")

    def schemas_for(self, granted_capabilities):
        """Return the schemas of tools whose capability is granted (least-privilege discovery)."""
        raise NotImplementedError("Complete filtered discovery")

    def dispatch(self, name, arguments, granted_capabilities):
        """Unknown name -> KeyError. Ungranted capability -> PermissionError. Otherwise call the handler."""
        raise NotImplementedError("Complete protected dispatch")

## Behavioural check

Run this after completing the starter cell. If you have not finished, it prints a hint instead of failing. A passing check proves the listed contract examples, not every possible input.

In [ ]:
def run_checks():
    registry = ToolRegistry()
    registry.register("add", {"name": "add", "parameters": {"a": "number", "b": "number"}},
                      lambda a, b: a + b, "math.read")
    print("visible with math.read :", [s["name"] for s in registry.schemas_for({"math.read"})])
    print("visible with nothing   :", [s["name"] for s in registry.schemas_for(set())])
    assert len(registry.schemas_for({"math.read"})) == 1
    assert registry.schemas_for(set()) == []
    assert registry.dispatch("add", {"a": 4, "b": 5}, {"math.read"}) == 9

    for label, call, expected in [
        ("ungranted dispatch", lambda: registry.dispatch("add", {"a": 1, "b": 1}, set()), PermissionError),
        ("unknown tool", lambda: registry.dispatch("nope", {}, {"math.read"}), KeyError),
        ("duplicate registration", lambda: registry.register("add", {}, lambda: None, "math.read"), ValueError),
    ]:
        try:
            call()
        except expected as exc:
            print(f"{label:<23} -> {type(exc).__name__}: {exc}")
        else:
            raise AssertionError(f"{label} must raise {expected.__name__}")
    print("PASS: registry centralizes discovery, dispatch, and capability checks")

try:
    run_checks()
except NotImplementedError:
    print("Not implemented yet. Complete the starter cell above, or study the reference solution below and re-run this cell.")

## Reference solution

Read this even if your check passed: compare each commented line with your version, then re-run the check cell above.

In [ ]:
# --- Reference solution: read it line by line, then re-run the check cell above ---
class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, name, schema, handler, capability):
        if name in self._tools:                                    # one name, one handler
            raise ValueError(f"tool {name!r} is already registered")
        self._tools[name] = {"schema": schema, "handler": handler, "capability": capability}

    def schemas_for(self, granted_capabilities):
        # Discovery is filtered so the model is never tempted by tools it may not use ...
        return [t["schema"] for t in self._tools.values() if t["capability"] in granted_capabilities]

    def dispatch(self, name, arguments, granted_capabilities):
        if name not in self._tools:                                # fail closed on unknown names
            raise KeyError(f"unknown tool {name!r}")
        tool = self._tools[name]
        if tool["capability"] not in granted_capabilities:         # ... and enforced AGAIN here
            raise PermissionError(f"capability {tool['capability']!r} not granted for {name!r}")
        return tool["handler"](**arguments)                        # only now does the host run it

print("Reference ToolRegistry defined. Re-run the check cell above to see PASS.")

## Explain

**Why must filtered schemas and protected dispatch both exist?**

<details><summary>Show answer</summary>

Filtering schemas reduces temptation: the model never sees a tool it may not use. But a model can still name a hidden tool, and code paths other than the model can call dispatch. Only the check inside dispatch actually prevents execution.

</details>

**Why raise an exception instead of returning None for an ungranted call?**

<details><summary>Show answer</summary>

A silent None can be mistaken for a successful empty result. An exception stops the run, is recorded as an event, and forces the caller to handle the denial explicitly.

</details>